# Reproducing Xiao, Severson et al. (2026) from dandiset 001687

Paper. Xiao W, Severson KS, Zheng H, Chen K, Thompson PM, Levy MS, Choi S, Zhao S, Takatoh J,
Prevosto V, Wang F (2026). Peri-head distance coding in the mouse brainstem. *Neuron*.
[doi:10.1016/j.neuron.2026.05.027](https://doi.org/10.1016/j.neuron.2026.05.027). The claims file
was written from the accepted author manuscript in PubMed Central (NIHMS2191201, PMC13336321) with
its supplement file (legends of Figures S1 to S6, Tables S2 to S5); the version of record was not read.

Dandiset. [DANDI:001687](https://dandiarchive.org/dandiset/001687/0.260821.2126), version
0.260821.2126. The paper cites no version. This is the most recent published version; it was
published with the release note "Added link to published article in Related Resources" and holds
the same 191 assets as 0.260805.1529. The earlier version 0.260618.1741 holds 67 assets (426 MB,
the processed files without the raw files). The analysis reads only the 67 processed files and,
for the whisker-cutting claims, the pass times of the raw files, so the latest version, which
also names the paper, is pinned.

What the dandiset holds. 67 processed silicon-probe sessions from 36 wild-type (C57) mice,
1,358 sorted PrV units, with spike times, the trials of the full-array wall passes (start time,
measured wall distance), the authors' per-unit labels and tuning curves; and the raw files
(broadband voltage, LFP, behaviour video, digital and analog sync lines, all wall-pass epochs).
It holds no tungsten recordings, no SpVi-TeLC or GtACR mice, no tracked whisker kinematics and
no anesthetized sessions. In the 25 whisker-cutting sessions the post-cut passes are in the raw
files' pass epochs but carry no wall distance.

Outside input. The paper's circuit model (`PrV_simulation`, GitHub
`wanglab-neuro/2026_Xiao-Severson_Peri-head-distance` at commit b1a9f3c), for the one
`elsewhere` claim.

Claims. Main reading for unit classification: the claims file, as amended after the question in
`questions/001687.yaml`, makes the responsiveness test with BH-FDR across the nine distances
(q < 0.05, as in Fig2DEF_Pop_Heatmap_Hist_Pie.m) the main reading and the uncorrected test
(p < 0.05 at any distance) the variant. An earlier run of this attempt judged the claims on the
uncorrected test, as the claims file then had it. Both readings are shown everywhere.

| claim | status here |
|---|---|
| figs1f-angle-awake-vs-anesthetized, figs1j-curvature-awake-vs-anesthetized, figs1d-bending-larger-closer | blocked: no tracked whiskers, no anesthetized sessions |
| figs2-tungsten-responsiveness, figs2-tungsten-tuning | blocked: no tungsten recordings |
| si-unit-yield | tested |
| fig2-wall-responsive-counts, fig2-distance-tuned-counts, fig2f-tuning-categories | tested |
| figs4a-baseline-activated-lower, figs4a-baseline-suppressed-similar, figs4b-activated-increase-suppressed-decrease, figs4c-untuned-no-change | tested |
| fig2de-preferred-distances-span, figs4ef-crossvalidated-stability, figs4l-temporal-windows-map | tested |
| figs4j-ventral-location | tested (peak channels from the raw voltage) |
| fig3a-decoding-above-controls, fig3-overall-decoding-accuracy, fig3c-decoding-by-distance, fig3c-decoding-error | tested |
| figs5c-single-neuron-decoders, figs5de-svm-robust | tested |
| fig4-cut-baseline-unchanged, fig4b-cut-wall-firing-reduced, fig4b-cut-per-unit-changes | blocked: the paper's stable units cannot be identified (results on all units of the cutting sessions shown) |
| fig4c-map-units-after-cut, fig4jk-proximity-units-after-cut, figs5fg-cut-decoding | blocked: post-cut passes have no wall distance |
| figs6-opto-local-inhibition-gain | blocked: no GtACR sessions |
| figs6i-telc-kinematics-unchanged, telc-unit-yield, fig5d-telc-preferred-closer, fig5e-telc-category-shift, fig5e-telc-untuned-ambiguous-unchanged, figs6j-telc-untuned-higher-firing, figs6k-telc-tuned-baseline-similar, figs6k-telc-tuned-wall-higher, figs6l-telc-increase-at-close-distances, fig6ghi-tuning-shape-telc | blocked: no SpVi-TeLC mice |
| fig6-model-telc-loses-map | tested on the paper's model code (`elsewhere`) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import tempfile
import collections
import concurrent.futures as cf
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import prv

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

t_start = time.time()
assets = prv.list_assets()
sessions = prv.load_all(assets)
print(f"dandiset {prv.DANDISET} version {prv.VERSION}: {len(assets)} assets; "
      f"loaded {len(sessions)} processed files in {time.time() - t_start:.0f} s")

## Data check

Rates are computed from the spike times and the trials table (counts in 3 to 11 s and -4.5 to 0 s
after each pass start), then checked against the tuning curves the deposit stores. Each check is
stated under its output.

In [ ]:
units = prv.unit_table(sessions)
spikes = [sp for s in sessions for sp in s["spikes"]]
meta = pd.DataFrame([s["meta"] for s in sessions])
meta["n_units"] = [len(s["spikes"]) for s in sessions]
meta["n_trials"] = [len(s["start"]) for s in sessions]
meta["cutting"] = meta.session_description.str.contains("cutting")
print("processed files:", len(meta), "; subjects:", meta.subject.nunique(), "; strains:", meta.strain.unique().tolist(),
      "; sorted units:", meta.n_units.sum())
print("sessions per subject:", meta.groupby("subject").size().value_counts().sort_index().to_dict())
print("whisker-cutting sessions (description names cutting):", int(meta.cutting.sum()),
      "from", meta[meta.cutting].subject.nunique(), "mice,", int(meta[meta.cutting].n_units.sum()), "units")
print("trial conditions present:", sorted({c for s in sessions for c in s["cond"]}))
print("probe named in subject description:", collections.Counter(p.split("Probe:")[-1].split(".")[0].strip() for p in meta.probe))

# distances
alld = np.concatenate([s["dist"] for s in sessions])
print(f"\nwall_distance_mm: {alld.min():.2f} to {alld.max():.2f} mm over {alld.size} passes; "
      f"outside 6-24 mm: {int(((alld < 6) | (alld >= 24)).sum())}")
nb = [len(np.unique(prv.dist_bin(s["dist"])[prv.dist_bin(s["dist"]) >= 0])) for s in sessions]
print("sessions by number of occupied 2-mm bins:", collections.Counter(nb))
reps = pd.Series([min(len(x) for x in u["starts"]) for u in units])
print("passes per bin (smallest bin of each unit): min", reps.min(), "median", reps.median(), "max", reps.max())
dur = np.concatenate([s["start"][1:] - s["start"][:-1] for s in sessions])
print(f"time between pass starts: min {dur.min():.1f} s, median {np.median(dur):.1f} s "
      "(the Methods: each pass about 14 s; windows reach 11.5 s after onset and 4.5 s before)")

# rates against the deposit's tuning curves
dev_w, dev_b = [], []
for s in sessions:
    for u in [x for x in units if x["path"] == s["meta"]["path"]]:
        mine = np.array([np.mean(x) if len(x) else np.nan for x in u["rates"]["wall"]])
        dep = np.asarray(s["dep_fr"][u["idx"]], float)
        if dep.size == 9:
            dev_w.append(np.nanmax(np.abs(mine - dep)))
        dev_b.append(abs(np.mean(np.concatenate(u["rates"]["base"])) - s["dep_base"][u["idx"]]))
bad = collections.Counter()
for s in sessions:
    for u in [x for x in units if x["path"] == s["meta"]["path"]]:
        mine = np.array([np.mean(x) if len(x) else np.nan for x in u["rates"]["wall"]])
        if np.nanmax(np.abs(mine - np.asarray(s["dep_fr"][u["idx"]], float))) > 1e-6:
            bad[u["session"]] += 1
print("units whose per-distance means differ from the deposit's, by session:", dict(bad))
print(f"\nlargest |own - deposit| per-distance mean wall rate over {len(dev_w)} units: {np.nanmax(dev_w):.2e} sp/s; "
      f"mean baseline: {np.nanmax(dev_b):.2e} sp/s")
wall_all = np.concatenate([np.concatenate(u["rates"]["wall"]) for u in units])
base_all = np.concatenate([np.concatenate(u["rates"]["base"]) for u in units])
print(f"per-pass wall rates 0 to {wall_all.max():.1f} sp/s, baseline 0 to {base_all.max():.1f} sp/s; "
      f"spike times {min(x.min() for x in spikes if len(x)):.1f} to {max(x.max() for x in spikes if len(x)):.0f} s")

For all but the units listed below, the deposit's stored tuning curves and baselines are the per-pass rates
computed here, to rounding, so the trials table, the spike times and the deposit's own analysis
share one clock and one pass selection. In the two sessions
with 1-mm steps (20221212 and 20221213) the deposit's nine values use one distance of each 2-mm
bin; here both distances of a bin are pooled, as the Methods' binning states ("uniformly binning
distance measurements into nine 2-mm-wide bins"). All 67 files are wild-type C57 mice with full arrays; the trial_condition column is
empty everywhere, so no file labels a post-cut, laser or TeLC pass.

In [ ]:
# Recordings deposited more than once: identical spike trains or identical pass times across files
sig = collections.defaultdict(list)
for s in sessions:
    for i, sp in enumerate(s["spikes"]):
        if len(sp) > 20:
            sig[prv.unit_signature(sp)].append((s["meta"]["session_id"], i))
dups = [v for v in sig.values() if len({x[0] for x in v}) > 1]
starts_sig = collections.defaultdict(list)
for s in sessions:
    starts_sig[tuple(np.round(s["start"][:20], 3))].append(s["meta"]["session_id"])
print("units whose first 50 spike times recur in another file:", len(dups))
print("files sharing their first 20 pass times:", [v for v in starts_sig.values() if len(v) > 1])
notes = meta[meta.session_description.str.contains("SAME", case=False)][["session_id", "session_description"]]
print("\nfiles whose notes say SAME RECORDING/SESSION:")
for _, r in notes.iterrows():
    print(" ", r.session_id, "|", r.session_description.split("Notes:")[-1].strip())
print("sessions named as the partners (session 1 of 20230209 and 20230226, sessions 3 of 20230226 and 20230329)",
      "deposited:", [x for x in meta.session_id if x in ("20230209_sp_session_1", "20230226_sp_session_1",
                                                          "20230226_sp_session_3", "20230329_sp_session_3")])

No unit's spike train and no session's pass times recur in another file, so no recording is
deposited twice. Four files say in their notes that they are the same recording as another
session; none of the named partners is deposited,
so each recording is counted once.

In [ ]:
# Unit-level measures: low-firing rule, ISI false alarms, and the paper's classification under both readings
t0 = time.time()
low = np.array([prv.low_firing(u) for u in units])
fa = np.array([prv.isi_fa(sp) for sp in spikes])
cls = [prv.classify(u) for u in units]
print(f"classified {sum(c is not None for c in cls)} of {len(units)} units in {time.time() - t0:.0f} s")
dep_low = np.array([u["low_firing"] for u in units])
print("low-firing (both rates < 1 Hz): own", int(low.sum()), "; deposit low_firing", int(dep_low.sum()),
      "; same units:", bool((low == dep_low).all()))
acc = (~low) & (fa <= 2)
print("ISI false-alarm rate > 2% (0.8 ms):", int((fa > 2).sum()), "units;", int(((fa > 2) & ~low).sum()), "of them not low-firing")
print("units passing ISI and firing criteria:", int(acc.sum()), "of", len(units), "(SNR is not computable: no waveforms in the processed files)")

U = pd.DataFrame({"session": [u["session"] for u in units], "subject": [u["subject"] for u in units],
                  "acc": acc, "low": low, "fa": fa,
                  "dep_type": [u["tuning_type"] for u in units], "dep_resp": [u["wall_responsive"] for u in units],
                  "tuned": [c["tuned"] for c in cls], "pref": [c["pref"] for c in cls], "trough": [c["trough"] for c in cls]})
for r in "mc":
    U[f"{r}_cat"] = [c[f"{r}_cat"] for c in cls]
    U[f"{r}_act"] = [c[f"{r}_act"] for c in cls]
    U[f"{r}_sup"] = [c[f"{r}_sup"] for c in cls]
U["base"] = [np.mean(np.concatenate(u["rates"]["base"])) for u in units]
U["cutting"] = U.session.isin(meta[meta.cutting].session_id)
NAMES = {"m": "uncorrected (variant)", "c": "BH-FDR (main)"}

# deposit labels against the main reading (BH-FDR)
dep = np.where(U.dep_resp | (U.dep_type != "untuned"), U.dep_type, "unresponsive")
dep = np.where(U.low, "low", dep)
own = np.where(U.low, "low", U.c_cat.replace({"suppressed_only": "suppressed"}))
print("\ndeposit labels (rows) against own classification, BH-FDR reading (columns), all 1,358 units:")
print(pd.crosstab(pd.Series(dep, name="deposit"), pd.Series(own, name="own (BH-FDR)")))
print("deposit wall_responsive units with ISI false alarms > 2%:", int((U.dep_resp & (U.fa > 2)).sum()))

The deposit's labels are reproduced unit for unit by the BH-FDR reading (the main reading), except for about
40 units the deposit calls unresponsive while they pass the tests here. 23 of these have ISI false
alarms above 2%, and no unit the deposit calls wall-responsive has, so the deposit's labels already
exclude the units that fail isolation, and the rest are consistent with the SNR criterion, which
cannot be applied here. The deposit has no column for the isolation criteria (SNR, false-alarm
rate) or for the paper's "acceptable" flag.

Clock check. The pass times and the spike times share a clock if the wall distance can be decoded
from the spikes at the true pass times but not when each pass is given the distance of another
pass. The map units of the BH-FDR reading are decoded below (MCC decoder of the decoding
section, 15 splits, one subsample).

In [ ]:
rng_clock = np.random.default_rng(11)
lefts = prv.unit_bins(spikes[0], units[0]["starts"])[1]
idx_map = np.where(U.acc & (U.c_cat == "map"))[0]
true_d = [prv.unit_bins(spikes[i], units[i]["starts"])[0] for i in idx_map]
acc_true = prv.summarize_cm(prv.decode(true_d, rng_clock, cv="15split", n_resample=1), lefts)[0].mean()
# give each pass the distance label of a pass at another distance: permute bins within each session
shift_d = []
for i in idx_map:
    u = units[i]
    perm = np.roll(np.arange(9), 4)
    shift_d.append(prv.unit_bins(spikes[i], [u["starts"][perm[k]] for k in range(9)])[0])
acc_shift = prv.summarize_cm(prv.decode(shift_d, rng_clock, cv="15split", n_resample=1), lefts)[0].mean()
lagged = []
for i in idx_map:
    u = units[i]
    lagged.append(prv.unit_bins(spikes[i], [np.asarray(st) + 30.0 for st in u["starts"]])[0])
acc_lag = prv.summarize_cm(prv.decode(lagged, rng_clock, cv="15split", n_resample=1), lefts)[0].mean()
print(f"{len(idx_map)} map units: accuracy 3-11 s with true labels {acc_true:.1f}%, labels moved by four bins "
      f"{acc_shift:.1f}%, pass times moved by +30 s {acc_lag:.1f}% (chance 11.1%)")

Distance is decoded far above chance at the true pass times. With the labels moved, accuracy is
again high because the permutation is the same for all units (a relabelling the decoder learns);
moving every pass time by 30 s, which puts the window in the next pass, drops accuracy toward
chance. The spike times and the trials table are consistent with one clock in every session that
contributes map units; sessions without map units are not checked by this test.

## si-unit-yield

> Silicon probe recordings in wild-type mice yielded 1,141 single units from 36 mice passing
> isolation and firing criteria, out of 1,358 sorted units.

Criteria from the claims file: SNR of at least 5, ISI false-alarm rate at most 2% at 0.8 ms, not
low-firing. SNR needs the units' waveforms, which the processed files do not hold; the other two
are applied.

In [ ]:
print("sorted units in the deposit:", len(U), "; mice:", U.subject.nunique())
print("not low-firing:", int((~U.low).sum()), "; and ISI false alarms <= 2%:", int(U.acc.sum()))
print("mice with at least one passing unit:", U[U.acc].subject.nunique())
print("passing units per mouse: min", U[U.acc].groupby("subject").size().min(), "max", U[U.acc].groupby("subject").size().max())
print("paper: 1141 of 1358 (36 mice); tolerance 1141 +/- 114.1")

**Result: reproduced, without the SNR criterion.** The deposit holds 1,358 sorted units from 36 mice. 1,188 are not low-firing and 1,160 of these also pass the ISI criterion, within tolerance of 1,141 (+/- 114.1); all 36 mice keep at least one unit. The SNR criterion cannot be applied, as the processed files hold no waveforms.

## fig2-wall-responsive-counts

> Of the 1,141 units, 768 (67.3%) were wall-responsive; of these 532 (69.3%) were activated at one or
> more distances, 333 (43.4%) suppressed, and 97 both.

Population: the 1,160 units passing the ISI and firing criteria. Main reading: one-tailed
signed-rank tests at each distance with BH-FDR across the nine distances, q < 0.05 (the claims
file, from Fig2DEF_Pop_Heatmap_Hist_Pie.m); variant: the same tests uncorrected, p < 0.05.

In [ ]:
A = U[U.acc]
def counts(r):
    resp = A[f"{r}_cat"] != "unresponsive"
    act, sup = A[f"{r}_act"], A[f"{r}_sup"]
    return dict(responsive=int(resp.sum()), activated=int(act.sum()), suppressed=int(sup.sum()), both=int((act & sup).sum()))
exp = dict(responsive=(768, 76.8), activated=(532, 53.2), suppressed=(333, 33.3), both=(97, 9.7))
tab = pd.DataFrame({"paper": {k: v[0] for k, v in exp.items()}, "tolerance": {k: v[1] for k, v in exp.items()},
                    "BH-FDR (main)": counts("c"), "uncorrected (variant)": counts("m")})
for col in ["BH-FDR (main)", "uncorrected (variant)"]:
    tab[col + " within"] = (tab[col] - tab.paper).abs() <= tab.tolerance
print(tab.to_string())
print("\nresponsive units per mouse (BH-FDR reading): median", A[A.c_cat != "unresponsive"].groupby("subject").size().median())
print("units excluded by the ISI criterion that would be responsive (uncorrected / BH-FDR):",
      int(((U.fa > 2) & ~U.low & (U.m_cat != "unresponsive")).sum()), "/", int(((U.fa > 2) & ~U.low & (U.c_cat != "unresponsive")).sum()))

**Result: reproduced.** With BH-FDR across the nine distances 782 units are responsive, 541 activated, 337 suppressed and 96 both, all within tolerance of 768, 532, 333 and 97. The uncorrected variant gives 1,061, 751, 657 and 347, all far outside tolerance.

## fig2-distance-tuned-counts

> Over the full wall-contact period, 656 of 768 wall-responsive units (85.4%) were distance tuned;
> of these, 472 (72.0%) were activated at one or more distances (the activated population), 213
> (32.5%) suppressed (the suppressed population), and 29 both.

In [ ]:
def tuned_counts(r):
    resp = A[f"{r}_cat"] != "unresponsive"
    t = resp & A.tuned
    act, sup = A[f"{r}_act"], A[f"{r}_sup"]
    return dict(tuned=int(t.sum()), activated_tuned=int((t & act).sum()), suppressed_tuned=int((t & sup).sum()),
                both_tuned=int((t & act & sup).sum()))
exp = dict(tuned=(656, 65.6), activated_tuned=(472, 47.2), suppressed_tuned=(213, 21.3), both_tuned=(29, 2.9))
tab = pd.DataFrame({"paper": {k: v[0] for k, v in exp.items()}, "tolerance": {k: v[1] for k, v in exp.items()},
                    "BH-FDR (main)": tuned_counts("c"), "uncorrected (variant)": tuned_counts("m")})
for col in ["BH-FDR (main)", "uncorrected (variant)"]:
    tab[col + " within"] = (tab[col] - tab.paper).abs() <= tab.tolerance
print(tab.to_string())

**Result: partially reproduced.** Under the BH-FDR reading 600 units are tuned, 478 activated and tuned and 214 suppressed and tuned, within tolerance of 656, 472 and 213, but 92 units are both activated and suppressed, against 29 +/- 2.9. The criterion asks for all four counts and gives no partial rule; three of the four hold. The uncorrected variant gives 658 tuned (within tolerance) and 568, 375 and 285 (outside).

## fig2f-tuning-categories

> Among the 768 wall-responsive units, 266 (34.6%) were proximity units, 118 (15.4%) map units, 88
> (11.5%) ambiguous, and 119 (15.5%) suppressed-only; the remainder (177, 23.0%) were untuned,
> giving a proximity to map ratio of 2.3 to 1.

Readings for the last two counts: (a) 119 and 177 as printed; (b) 184 and 112.

In [ ]:
cats = ["proximity", "map", "ambiguous", "suppressed_only", "untuned"]
exp_a = dict(proximity=266, map=118, ambiguous=88, suppressed_only=119, untuned=177)
exp_b = dict(exp_a, suppressed_only=184, untuned=112)
rows = []
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    vc = A[f"{r}_cat"].value_counts()
    for k in cats:
        n = int(vc.get(k, 0))
        rows.append(dict(reading=name, category=k, count=n, paper_a=exp_a[k], within_a=abs(n - exp_a[k]) <= 0.1 * exp_a[k],
                         paper_b=exp_b[k], within_b=abs(n - exp_b[k]) <= 0.1 * exp_b[k]))
    print(f"{name}: proximity / map ratio {vc.get('proximity', 0) / vc.get('map', 1):.2f}")
print(pd.DataFrame(rows).to_string(index=False))
print("\ndeposit tuning_type among deposit wall_responsive units:", U[U.dep_resp].dep_type.value_counts().to_dict())

**Result: reproduced under reading (a).** Under the BH-FDR reading the five counts are 271 proximity, 120 map, 87 ambiguous, 122 suppressed-only and 182 untuned, all within tolerance of the Figure 2F values, with a proximity to map ratio of 2.26; reading (b) fails on the last two (184 and 112). The uncorrected variant gives 303, 168, 97, 90 and 403, all outside tolerance. The deposit's own labels give 267, 118, 87, 119 and 177.

## figs4a-baseline-activated-lower

> Before wall movement, units of the activated population fire less than untuned units (5.46
> against 6.93 sp/s).

Per unit, the mean baseline rate over all passes; two-tailed two-sample K-S test; mean +/- the
bootstrap 95% CI half-width (Methods, 10,000 resamples).

In [ ]:
rng_s4a = np.random.default_rng(401)
def pops(r):
    resp = A[f"{r}_cat"] != "unresponsive"
    return dict(activated=A[resp & A.tuned & A[f"{r}_act"]], suppressed=A[resp & A.tuned & A[f"{r}_sup"]],
                untuned=A[resp & ~A.tuned])
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    P = pops(r)
    a, un = P["activated"].base.values, P["untuned"].base.values
    ks = stats.ks_2samp(a, un)
    print(f"{name}: activated n={a.size} {a.mean():.2f} +/- {prv.boot_ci_half(a, rng_s4a):.3f} sp/s; "
          f"untuned n={un.size} {un.mean():.2f} +/- {prv.boot_ci_half(un, rng_s4a):.3f} sp/s; "
          f"K-S D={ks.statistic:.3f} p={ks.pvalue:.2e}; medians {np.median(a):.2f} vs {np.median(un):.2f}")
    for subj_n in [P["activated"].groupby("subject").base.mean().describe()[["count", "min", "max"]].round(2).to_dict()]:
        print("   activated, per-mouse means:", subj_n)
print("paper: 5.46 +/- 0.670 vs 6.93 +/- 0.947 sp/s, p = 2.47e-7")

**Result: reproduced.** Under the BH-FDR reading the 478 activated units fire 5.40 sp/s at baseline against 6.97 sp/s for the 182 untuned units (two-tailed K-S p = 2.90e-07), both means within tolerance of 5.46 and 6.93; activated units come from 35 mice. The uncorrected variant gives 5.62 against 6.71 sp/s (568 and 403 units, p = 1.81e-07).

## figs4a-baseline-suppressed-similar

> Units of the suppressed population have baseline firing comparable to untuned units (7.31 against
> 6.93 sp/s, not significant).

A claim of absence: two-tailed two-sample K-S test of per-unit mean baseline rates, suppressed
population against untuned units, at p < 0.05.

In [ ]:
rng_s4a2 = np.random.default_rng(402)
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    P = pops(r)
    sp_, un = P["suppressed"].base.values, P["untuned"].base.values
    ks = stats.ks_2samp(sp_, un)
    print(f"{name}: suppressed n={sp_.size} {sp_.mean():.2f} +/- {prv.boot_ci_half(sp_, rng_s4a2):.3f} sp/s; untuned n={un.size} "
          f"{un.mean():.2f} sp/s; difference {sp_.mean() - un.mean():+.2f} sp/s; K-S D={ks.statistic:.3f} p={ks.pvalue:.3g}")
print("paper: 7.31 +/- 1.124 vs 6.93 +/- 0.947 sp/s, p = 0.63")

**Result: reproduced.** Under the BH-FDR reading the K-S test of the 214 suppressed against the 182 untuned units gives p = 0.643, so no difference at 0.05 (7.25 against 6.97 sp/s, difference +0.29 sp/s; 7.25 is within tolerance of 7.31). The uncorrected variant gives p = 0.0944 (6.71 and 6.71 sp/s).

## figs4b-activated-increase-suppressed-decrease

> During wall passes the activated population fires more than at baseline (16.44 against 6.36 sp/s
> at the preferred distance) and the suppressed population fires less (4.47 against 6.19 sp/s).

Per unit, the mean wall rate at the preferred distance (activated) or at the distance of minimum
rate (suppressed), against baseline: (a) of the passes at that distance, (b) of all passes.
One-tailed two-sample K-S tests in the stated direction.

In [ ]:
rng_s4b = np.random.default_rng(403)
def at_bin(i, k, w):
    return float(np.mean(units[i]["rates"][w][k]))
rows = []
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    P = pops(r)
    for popname, col, alt in [("activated", "pref", "less"), ("suppressed", "trough", "greater")]:
        idx = P[popname].index.values
        wall = np.array([at_bin(i, U.loc[i, col], "wall") for i in idx])
        base_a = np.array([at_bin(i, U.loc[i, col], "base") for i in idx])
        base_b = U.loc[idx, "base"].values
        for reading, base in [("a: same-distance baseline", base_a), ("b: all-pass baseline", base_b)]:
            ks = stats.ks_2samp(wall, base, alternative=alt)
            rows.append(dict(classification=name, population=popname, n=idx.size, reading=reading,
                             wall=round(wall.mean(), 2), wall_ci=round(prv.boot_ci_half(wall, rng_s4b), 3),
                             baseline=round(base.mean(), 2), base_ci=round(prv.boot_ci_half(base, rng_s4b), 3),
                             ks_D=round(ks.statistic, 3), p=f"{ks.pvalue:.2e}"))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: activated 16.44 +/- 1.640 vs 6.36 +/- 0.704 (p = 9.26e-28); suppressed 4.47 +/- 1.000 vs 6.19 +/- 1.076 (p = 3.06e-4)")
print("K-S direction: scipy alternative 'less' = first sample (wall) stochastically larger; 'greater' = smaller")

**Result: partially reproduced.** Both one-tailed K-S tests hold under both baseline readings and both classifications (activated p at most 6.2e-37, suppressed p at most 2.3e-08). Under the BH-FDR reading the activated wall rate (16.67 sp/s) and the suppressed wall rate (4.34 sp/s) are within tolerance, and the suppressed baseline is within tolerance with the all-pass baseline (7.25 sp/s, reading b) but not with the same-distance baseline (7.32 sp/s, reading a); the activated baseline, 5.36 or 5.40 sp/s, is outside 6.36 +/- 0.704 under both readings. The uncorrected variant also leaves the activated baseline out (5.60 or 5.62 sp/s). The paper's 6.36 sp/s differs from its own 5.46 sp/s for the same population's baseline (figs4a-baseline-activated-lower), which these numbers match.

## figs4c-untuned-no-change

> Untuned units do not change their firing between baseline and wall passes (6.93 against 6.32 sp/s).

A claim of absence: two-tailed two-sample K-S test between per-unit mean baseline rates and
per-unit wall rates averaged across distances, untuned units, at p < 0.05.

In [ ]:
rng_s4c = np.random.default_rng(404)
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    idx = pops(r)["untuned"].index.values
    wall = np.array([np.mean([np.mean(x) for x in units[i]["rates"]["wall"]]) for i in idx])
    base = U.loc[idx, "base"].values
    ks = stats.ks_2samp(base, wall)
    print(f"{name}: untuned n={idx.size}; baseline {base.mean():.2f} +/- {prv.boot_ci_half(base, rng_s4c):.3f}, "
          f"wall {wall.mean():.2f} +/- {prv.boot_ci_half(wall, rng_s4c):.3f} sp/s; difference {wall.mean() - base.mean():+.2f}; "
          f"K-S D={ks.statistic:.3f} p={ks.pvalue:.3g}")
print("paper: 6.93 +/- 0.947 vs 6.32 +/- 0.953 sp/s, p = 0.53; tolerance on the wall mean 0.953")

**Result: reproduced.** Under the BH-FDR reading the 182 untuned units fire 6.97 sp/s at baseline and 6.36 sp/s during passes (difference -0.60 sp/s, two-tailed K-S p = 0.484), the wall mean within tolerance of 6.32. The uncorrected variant gives 6.71 and 6.41 sp/s (403 units, p = 0.645).

## fig2de-preferred-distances-span

> In both the activated and the suppressed populations a large proportion of units prefer the
> closest wall position (7 mm), while others are best tuned to intermediate or far distances.

Preferred distance: maximum of the mean tuning curve (activated), minimum (suppressed).

In [ ]:
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    P = pops(r)
    for popname, col in [("activated", "pref"), ("suppressed", "trough")]:
        h = np.bincount(P[popname][col].values, minlength=9)
        print(f"{name}, {popname} (n={h.sum()}): " + ", ".join(f"{d} mm {n}" for d, n in zip(prv.CENTERS, h))
              + f" | most frequent {prv.CENTERS[h.argmax()]} mm ({100 * h.max() / h.sum():.0f}%), "
              f"distances 9-23 mm with >= 1 unit: {int((h[1:] > 0).sum())} of 8")
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5))
for a_, (popname, col) in zip(ax, [("activated", "pref"), ("suppressed", "trough")]):
    a_.bar(prv.CENTERS, np.bincount(pops("m")[popname][col].values, minlength=9), width=1.6, label="uncorrected")
    a_.bar(prv.CENTERS, np.bincount(pops("c")[popname][col].values, minlength=9), width=0.8, label="BH-FDR")
    a_.set_title(popname); a_.set_xlabel("preferred distance (mm)")
ax[0].set_ylabel("units"); ax[0].legend(); plt.tight_layout(); plt.show()

**Result: reproduced.** In both populations and under both readings 7 mm is the most frequent preferred distance (BH-FDR: activated 57% of 478 units, suppressed trough 37% of 214; uncorrected: 53% and 29%), and each distance from 9 to 23 mm is preferred by at least one unit.

## figs4ef-crossvalidated-stability

> Preferred distances of the activated population are stable across two random halves of the wall
> passes, whereas the suppressed population is only partly reproducible, with only
> proximity-preferring responses stable.

Per unit, floor(reps/2) repetition indices drawn without replacement, the same for all distances;
preferred distance (max for activated, min for suppressed) in each half; (a) fraction with the same
preferred distance, (b) Spearman correlation across halves; 100 splits, median reported.

In [ ]:
rng_s4ef = np.random.default_rng(405)
def halves(i, use_min, rng):
    w, m = prv.equalize(units[i]["rates"]["wall"])
    W = np.stack(w, 0)                       # (9, reps)
    sel = np.zeros(m, bool); sel[rng.choice(m, m // 2, replace=False)] = True
    h1, h2 = W[:, sel].mean(1), W[:, ~sel].mean(1)
    f = np.argmin if use_min else np.argmax
    return f(h1), f(h2)
res = []
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    P = pops(r)
    for rep in range(100):
        out = {}
        for popname, use_min in [("activated", False), ("suppressed", True)]:
            pr = np.array([halves(i, use_min, rng_s4ef) for i in P[popname].index.values])
            out[popname + "_same"] = np.mean(pr[:, 0] == pr[:, 1])
            out[popname + "_rho"] = stats.spearmanr(pr[:, 0], pr[:, 1]).statistic
            if use_min:
                out["sup_keep_7"] = np.mean(pr[pr[:, 0] == 0, 1] == 0)
                out["sup_keep_far"] = np.mean(pr[pr[:, 0] > 0, 1] == pr[pr[:, 0] > 0, 0])
        res.append(dict(reading=name, **out))
R = pd.DataFrame(res)
print(R.groupby("reading").median().round(3).T.to_string())
print("\nrange over the 100 splits:")
print(R.groupby("reading").agg(["min", "max"]).round(3).T.to_string())

**Result: partially reproduced.** Under the BH-FDR reading, over 100 random splits, the activated population keeps its preferred distance more often than the suppressed population (median fraction 0.724 against 0.507), and suppressed units with a first-half trough at 7 mm keep it in 0.897 of cases against 0.293 for troughs at 9 mm or farther. But the Spearman correlation across halves is lower for the activated population (0.815 against 0.831; ranges over splits 0.768 to 0.855 and 0.771 to 0.890), so reading (b) fails and the criterion, which asks for both readings, holds in part. Under the uncorrected variant both readings hold (0.678 against 0.457; 0.827 against 0.779).

## figs4l-temporal-windows-map

> Applied to temporal windows, the classification finds map units most often in the sustained
> period (122 of 768, 15.9%), less in the onset period (83, 10.8%) and fewest in the offset period
> (34, 4.4%).

Readings: (a) responsiveness kept from the full period, tuning and map tests rerun in each window;
(b) all tests rerun in each window.

In [ ]:
t0 = time.time()
win = {}
for w in ["onset", "sustained", "offset"]:
    win[w] = [prv.classify(units[i], window=w) for i in A.index]
print(f"window classifications in {time.time() - t0:.0f} s")
rows = []
exp = dict(sustained=(122, 12.2), onset=(83, 8.3), offset=(34, 3.4))
for r, name in [("m", "uncorrected (variant)"), ("c", "BH-FDR (main)")]:
    full_act = A[f"{r}_act"].values
    full_resp = (A[f"{r}_cat"] != "unresponsive").values
    for w in ["sustained", "onset", "offset"]:
        C = win[w]
        map_a = sum(fa_ and c["tuned"] and c["pref"] > 0 and c["min_q_map"] < 0.05 for fa_, c in zip(full_act, C))
        map_b = sum(c[f"{r}_cat"] == "map" for c in C)
        map_b_resp = sum(c[f"{r}_cat"] == "map" and fr for c, fr in zip(C, full_resp))
        rows.append(dict(classification=name, window=w, paper=exp[w][0], tol=exp[w][1], a=map_a, b=map_b,
                         b_within_full_responsive=map_b_resp,
                         a_within=abs(map_a - exp[w][0]) <= exp[w][1], b_within=abs(map_b - exp[w][0]) <= exp[w][1]))
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** The order sustained > onset > offset holds under both readings and both classifications. Under the BH-FDR reading the sustained and onset counts are within tolerance (133 and 90 for reading a, 130 and 86 for b, against 122 and 83) and the offset count is not (68 or 42, against 34 +/- 3.4). The uncorrected variant gives 182, 109 and 96 (a) and 182, 104 and 84 (b), all outside tolerance.

## figs4j-ventral-location

> Wall-responsive units, and proximity and map units among them, lie preferentially in ventral PrV,
> toward the probe tip.

Depth: the shank coordinate (from the tip) of the unit's peak channel, the channel of largest
peak-to-trough amplitude of the mean waveform (Methods); Poly2 probes only (the raw files' electrode
group). The processed files hold no waveforms, so the mean waveform is computed here from the raw
voltage, over the spikes in 16 storage chunks (5.2 s each) spread over each session's wall passes
(see `prv.unit_peak_depths`). Readings: (a) median depths; (b) fractions in the 0-400 and 400-800 um
halves.

In [ ]:
t0 = time.time()
def depth_job(s):
    raw = prv.raw_asset_for(assets, s["meta"]["path"])
    return prv.unit_peak_depths(raw, s["spikes"], s["start"].min(), s["start"].max() + 11.0)
with cf.ThreadPoolExecutor(4) as ex:
    DEP = list(ex.map(depth_job, sessions))
U["probe_group"] = np.concatenate([[g] * len(o) for g, o in DEP])
U["depth"] = np.concatenate([[x["depth_um"] for x in o] for g, o in DEP])
U["depth_spikes"] = np.concatenate([[x["n_spikes"] for x in o] for g, o in DEP])
print(f"peak channels from {len(DEP)} raw files in {time.time() - t0:.0f} s")
print("electrode groups:", U.groupby("probe_group").size().to_dict())
poly2 = U.acc & U.probe_group.str.contains("poly2")
print("Poly2 units passing criteria:", int(poly2.sum()), "; with a depth (>= 20 spikes in the chunks):", int((poly2 & U.depth.notna()).sum()))
print("paper (Figure S4I-J legends): n = 1050 well-isolated and 703 wall-responsive Poly2 units")
for r in "mc":
    B = U[poly2 & U.depth.notna()]
    resp = B[f"{r}_cat"] != "unresponsive"
    unresp = B[~resp].depth
    pm = B[B[f"{r}_cat"].isin(["proximity", "map"])]
    med = dict(responsive=B[resp].depth.median(), unresponsive=unresp.median(),
               proximity=B[B[f"{r}_cat"] == "proximity"].depth.median(), map=B[B[f"{r}_cat"] == "map"].depth.median())
    mw = stats.mannwhitneyu(B[resp].depth, unresp)
    lo, hi = B.depth < 400, B.depth >= 400
    fr = dict(resp_lo=resp[lo].mean(), resp_hi=resp[hi].mean(),
              pm_lo=B[lo][f"{r}_cat"].isin(["proximity", "map"]).mean(), pm_hi=B[hi][f"{r}_cat"].isin(["proximity", "map"]).mean())
    a_ok = med["responsive"] < med["unresponsive"] and med["proximity"] < med["unresponsive"] and med["map"] < med["unresponsive"]
    b_ok = fr["resp_lo"] > fr["resp_hi"] and fr["pm_lo"] > fr["pm_hi"]
    print(f"\n{NAMES[r]}: n={len(B)}, responsive {int(resp.sum())}, unresponsive {int((~resp).sum())}")
    print("  median depth (um):", {k: float(v) for k, v in med.items()}, f"| rank-sum responsive vs unresponsive p={mw.pvalue:.3g}")
    print("  fractions, 0-400 vs 400-800 um:", {k: round(v, 3) for k, v in fr.items()}, "| units per half:", int(lo.sum()), int(hi.sum()))
    print("  reading a holds:", a_ok, "| reading b holds:", b_ok)
    h = pd.crosstab(pd.cut(B.depth, np.arange(0, 900, 100), right=False), B[f"{r}_cat"])
    print(h.to_string())

**Result: reproduced.** Of 1,082 Poly2 units passing the criteria (the paper's legend gives 1,050), 1,075 have a peak channel. Under the BH-FDR reading 720 are responsive (the legend gives 703). The median depth of wall-responsive units is smaller than that of unresponsive units (BH-FDR 350 against 525 um, rank-sum p = 3.85e-25; uncorrected 400 against 525 um), and so are the medians of proximity and map units (275 and 250 um; 300 and 300 um). The fraction of units that are responsive is higher in the 0-400 um half than in the 400-800 um half (0.834 against 0.534; 0.955 against 0.872), and so is the fraction of proximity or map units (0.517 against 0.184; 0.577 against 0.26). Both readings agree under both classifications.

## Population decoding (claims fig3a, fig3, fig3c, figs5de)

From the claims file: maximum-correlation-coefficient classifier on a pseudo-population; spike
counts in 150-ms windows stepped by 50 ms from 0 to 14 s; 15 passes per distance drawn 5 times;
z-scored features (training-set mean and SD); subpopulations of 118 units drawn 10 times without
replacement (combined: 36 map and 82 proximity units; a group with fewer than 118 units is used
whole each time); a shuffled group drawn from map and proximity units with each unit's passes
permuted across distances; accuracy per distance is the true-positive rate over time bins starting
within 3 to 11 s; overall accuracy is the mean over the nine distances. Cross-validation: (x)
10-fold (Methods, main), (y) 15 splits, one pass per distance held out (code). Train and test at
the same time bin (the Neural Decoding Toolbox default the code uses). Only the bins starting in
3 to 11 s are decoded, since nothing outside them enters a claim.

In [ ]:
t0 = time.time()
lefts_all = prv.unit_bins(spikes[0], units[0]["starts"])[1]
keep = (lefts_all >= 3 - 1e-9) & (lefts_all <= 11 + 1e-9)
lefts = lefts_all[keep]
D = {i: [x[:, keep] for x in prv.unit_bins(spikes[i], units[i]["starts"])[0]] for i in A.index}
print(f"binned {len(D)} units, {keep.sum()} time bins starting at {lefts[0]:.2f} to {lefts[-1]:.2f} s, in {time.time() - t0:.0f} s")
print("units with fewer than 15 passes at some distance:", sum(min(x.shape[0] for x in D[i]) < 15 for i in D))

GROUPS = ["map", "proximity", "mixed", "suppressed_only", "untuned", "shuffled"]
def group_units(r):
    cat = A[f"{r}_cat"]
    g = {k: cat.index[cat == k].values for k in ["map", "proximity", "suppressed_only", "untuned"]}
    g["shuffled_pool"] = np.concatenate([g["map"], g["proximity"]])
    return g
def draw(g, name, rng, size=118):
    if name == "mixed":
        return np.concatenate([rng.choice(g["map"], 36, replace=False), rng.choice(g["proximity"], 82, replace=False)])
    pool = g["shuffled_pool"] if name == "shuffled" else g[name]
    return rng.choice(pool, min(size, pool.size), replace=False)
for r in "mc":
    g = group_units(r)
    print(r, {k: v.size for k, v in g.items()})

rng_dec = np.random.default_rng(301)
CM = {}
t0 = time.time()
for r in "mc":
    g = group_units(r)
    for cv in ["10fold", "15split"]:
        for name in GROUPS:
            CM[(r, cv, name)] = [prv.decode([D[i] for i in draw(g, name, rng_dec)], rng_dec, cv=cv,
                                            shuffle=(name == "shuffled")) for _ in range(10)]
print(f"MCC decoding, 2 classifications x 2 cross-validations x 6 groups x 10 subsamples, in {time.time() - t0:.0f} s")

def summary(r, cv, name, t0=3.0, t1=11.0, thin=1):
    L = lefts[::thin]
    out = [prv.summarize_cm(c[:, :, ::thin], L, t0, t1) for c in CM[(r, cv, name)]]
    return np.array([o[0] for o in out]), np.array([o[1] for o in out])   # (10, 9) accuracy %, error mm
def ranksum(x, y):
    return stats.mannwhitneyu(x, y, alternative="two-sided", method="asymptotic", use_continuity=True).pvalue
NAMES = {"m": "uncorrected (variant)", "c": "BH-FDR (main)"}

## fig3a-decoding-above-controls

> Population decoders trained on map, proximity, suppressed-only and combined map plus proximity
> units decode wall distance during the contact period better than shuffled and untuned controls.

In [ ]:
rows = []
for r in "mc":
    for cv in ["10fold", "15split"]:
        ov = {n: summary(r, cv, n)[0].mean(1) for n in GROUPS}
        for n in ["map", "proximity", "mixed", "suppressed_only"]:
            for ctrl in ["shuffled", "untuned"]:
                rows.append(dict(classification=NAMES[r], cv=cv, group=n, control=ctrl, group_acc=round(ov[n].mean(), 2),
                                 control_acc=round(ov[ctrl].mean(), 2), p=f"{ranksum(ov[n], ov[ctrl]):.2e}",
                                 holds=bool(ov[n].mean() > ov[ctrl].mean() and ranksum(ov[n], ov[ctrl]) < 0.05)))
T3a = pd.DataFrame(rows)
print(T3a.to_string(index=False))
print("\nall pairs hold, per classification and cross-validation:")
print(T3a.groupby(["classification", "cv"]).holds.all().to_string())

**Result: reproduced.** Under both classifications and both cross-validations the map, proximity, combined and suppressed-only groups exceed both the shuffled group (10.99 to 11.26%) and the untuned group (15.25 to 16.55%), each pair with rank-sum p = 1.83e-04 (complete separation of the 10 subsamples). Under the BH-FDR classification with 10-fold the four groups reach 70.44, 48.37, 60.90 and 33.69%.

## fig3-overall-decoding-accuracy

> Averaged across the nine distances, the map group decodes best (71.5%), ahead of the combined map
> plus proximity group (58.6%) and the proximity group (48.9%); the suppressed-only group reaches
> 32.6% and untuned units 15.7%, against a chance level of 11.1%.

Readings: window (i) 3 to 11 s, (ii) 4.5 to 9.5 s; cross-validation (x) 10-fold, (y) 15 splits.

In [ ]:
EXP3 = dict(map=(71.53, 7.15), mixed=(58.55, 5.86), proximity=(48.86, 4.89), suppressed_only=(32.56, 3.26),
            untuned=(15.73, 1.57), shuffled=(10.9, 1.09))
order = ["map", "mixed", "proximity", "suppressed_only", "untuned", "shuffled"]
rows = []
for r in "mc":
    for cv in ["10fold", "15split"]:
        for wname, (a, b) in [("i 3-11 s", (3, 11)), ("ii 4.5-9.5 s", (4.5, 9.5))]:
            ov = {n: summary(r, cv, n, a, b)[0].mean(1) for n in GROUPS}
            means = [ov[n].mean() for n in order]
            within = [abs(ov[n].mean() - EXP3[n][0]) <= EXP3[n][1] for n in order]
            rows.append(dict(classification=NAMES[r], cv=cv, window=wname, **{n: round(ov[n].mean(), 2) for n in order},
                             order_holds=bool(np.all(np.diff(means) < 0)), n_within=int(sum(within)), all_within=all(within)))
print("paper:", {n: EXP3[n][0] for n in order})
print(pd.DataFrame(rows).to_string(index=False))
sd = {n: summary("c", "10fold", n)[0].mean(1).std(ddof=1) for n in order}
print("\nSD over the 10 subsamples, BH-FDR classification, 10-fold, 3-11 s:", {k: round(v, 2) for k, v in sd.items()})
print("chance 100/9 =", round(100 / 9, 2))

**Result: reproduced.** Under the BH-FDR classification the order map > combined > proximity > suppressed-only > untuned > shuffled holds under all four readings, and all six values are within tolerance with window (i) 3 to 11 s under both cross-validations (10-fold: 70.44, 60.90, 48.37, 33.69, 16.38 and 11.04%; 15 splits: 70.65, 62.64, 47.64, 33.81, 16.55 and 11.15%). With window (ii) 4.5 to 9.5 s the map value is above its tolerance (78.62 and 78.79%, against 71.53 +/- 7.15). Under the uncorrected variant the order also holds everywhere and all six values are within tolerance with 15 splits over 3 to 11 s and with 10-fold over 4.5 to 9.5 s; with 10-fold over 3 to 11 s the map value (63.09%) is below its tolerance.

## fig3c-decoding-by-distance

> At the two closest distances (7 and 9 mm) the combined map plus proximity and proximity groups
> decode most accurately, whereas the map group is most accurate at 11 to 23 mm.

In [ ]:
rows = []
for r in "mc":
    for cv in ["10fold", "15split"]:
        acc = {n: summary(r, cv, n)[0] for n in GROUPS}          # (10, 9)
        m = {n: acc[n].mean(0) for n in GROUPS}
        a_ok = all(m["mixed"][k] > m["map"][k] and m["proximity"][k] > m["map"][k] for k in [0, 1])
        q_mp = prv.bh([ranksum(acc["map"][:, k], acc["proximity"][:, k]) for k in range(9)])
        q_mm = prv.bh([ranksum(acc["map"][:, k], acc["mixed"][:, k]) for k in range(9)])
        b_best = [all(m["map"][k] > m[n][k] for n in GROUPS if n != "map") for k in range(2, 9)]
        b_q = [q_mp[k] < 0.05 and q_mm[k] < 0.05 for k in range(2, 9)]
        vals = dict(mixed7=m["mixed"][0], prox7=m["proximity"][0], map7=m["map"][0], map15=m["map"][4], prox15=m["proximity"][4])
        exp = dict(mixed7=(88.48, 8.85), prox7=(87.84, 8.78), map7=(81.14, 8.11), map15=(82.5, 8.25), prox15=(36.29, 3.63))
        within = {k: abs(vals[k] - exp[k][0]) <= exp[k][1] for k in exp}
        rows.append(dict(classification=NAMES[r], cv=cv, a_7_9=a_ok, b_map_best_11_23=f"{sum(b_best)}/7",
                         b_q_lt_005_11_23=f"{sum(b_q)}/7", **{k: round(v, 1) for k, v in vals.items()},
                         n_within=sum(within.values())))
        if r == "c":
            print(f"per-distance accuracy (%), BH-FDR classification, {cv}:")
            print(pd.DataFrame({n: np.round(m[n], 1) for n in GROUPS}, index=prv.CENTERS).T.to_string())
            print("BH q map vs proximity:", np.round(q_mp, 4).tolist())
            print("BH q map vs combined: ", np.round(q_mm, 4).tolist())
print("\npaper: combined 7 mm 88.48, proximity 7 mm 87.84, map 7 mm 81.14, map 15 mm 82.5, proximity 15 mm 36.29")
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Under the BH-FDR classification the map group is best at each distance from 11 to 23 mm, with BH-corrected rank-sum q < 0.05 against both the proximity and the combined group at all seven, under both cross-validations. At 7 mm the combined (94.0%) and proximity (92.3%) groups exceed the map group (79.6%), but at 9 mm the proximity group (79.4%) is below the map group (81.5%) with 10-fold, so part (a) fails at one distance; with 15 splits part (a) fails too, and since it holds at 7 mm (95.0 and 92.3% against 80.2%) it fails at 9 mm. Four of the five values are within tolerance; the proximity group at 15 mm is not (32.0% with 10-fold, 31.3% with 15 splits, against 36.29 +/- 3.63). This is the partial rule of the criterion. Under the uncorrected variant (a) and (b) hold and three of the five values are within tolerance (map 72.2% and proximity 29.1% at 15 mm are not).

## fig3c-decoding-error

> Decoding error grows with distance; proximity, map and combined groups reach submillimetre error at
> 7 and 9 mm, and only the map group is submillimetre at 13 and 15 mm.

Error per true distance: the confusion-weighted mean absolute difference between predicted and true
distance, averaged over the time bins starting in 3 to 11 s.

In [ ]:
rows = []
for r in "mc":
    for cv in ["10fold", "15split"]:
        err = {n: summary(r, cv, n)[1].mean(0) for n in GROUPS}
        c1 = all(err[n][k] < 1 for n in ["proximity", "map", "mixed"] for k in [0, 1])
        c2 = all(err["map"][k] < 1 for k in [3, 4]) and all(err[n][k] >= 1 for n in ["proximity", "mixed"] for k in [3, 4])
        vals = dict(map13=err["map"][3], map15=err["map"][4], map_mean=err["map"].mean(), prox_mean=err["proximity"].mean())
        exp = dict(map13=(0.70, 0.07), map15=(0.78, 0.078), map_mean=(0.86, 0.086), prox_mean=(1.78, 0.178))
        within = {k: abs(vals[k] - exp[k][0]) <= exp[k][1] for k in exp}
        rows.append(dict(classification=NAMES[r], cv=cv, sub_mm_7_9=c1, only_map_sub_mm_13_15=c2,
                         **{k: round(v, 3) for k, v in vals.items()}, n_within=sum(within.values()), all_within=all(within.values())))
        if r == "c":
            print(f"per-distance error (mm), BH-FDR classification, {cv}:")
            print(pd.DataFrame({n: np.round(err[n], 2) for n in GROUPS}, index=prv.CENTERS).T.to_string())
print("\npaper: map 13 mm 0.70, map 15 mm 0.78, map mean 0.86, proximity mean 1.78 mm")
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under the BH-FDR classification, with either cross-validation, the proximity, map and combined errors are below 1 mm at 7 and 9 mm, only the map group is below 1 mm at 13 and 15 mm, and the four values are within tolerance (10-fold 0.737, 0.830, 0.894 and 1.822 mm; 15 splits 0.726, 0.832, 0.888 and 1.879 mm, against 0.70, 0.78, 0.86 and 1.78). Under the uncorrected variant the map error is not below 1 mm at 13 and 15 mm with 10-fold (1.017 and 1.062 mm) and only the proximity mean error is within tolerance.

## figs5c-single-neuron-decoders

> Single-neuron decoders show the same pattern with lower accuracy than population decoders,
> proximity units being more accurate than map units at 7 mm and map units more accurate at 11 to
> 17 mm.

Per unit: spike counts in 200-ms bins over 3 to 11 s, a mean template per distance, nearest
template in Euclidean distance, leave-two-out exhaustively (the same two pass indices held out at
every distance), passes truncated to the smallest count across distances. Proximity against map
per distance, Wilcoxon rank-sum, BH-FDR over the nine distances.

In [ ]:
t0 = time.time()
need = set()
for r in "mc":
    need |= set(A.index[A[f"{r}_cat"].isin(["map", "proximity", "suppressed_only", "untuned"])])
SN = {}
for i in sorted(need):
    cb = [prv.binned_counts(spikes[i], np.asarray(st), 3.0, 11.0, prv.SN_BIN, prv.SN_BIN)[0] for st in units[i]["starts"]]
    cb, _ = prv.equalize(cb)
    SN[i] = prv.single_neuron_decoder(cb)
print(f"single-neuron decoders for {len(SN)} units in {time.time() - t0:.0f} s")
rng_sn = np.random.default_rng(501)
EXP_SN = dict(prox7=(65.84, 2.582), map7=(59.08, 3.702), map15=(34.72, 4.463), prox15=(18.84, 2.154))
for r in "mc":
    g = group_units(r)
    acc = {n: np.array([SN[i] for i in g[n]]) for n in ["map", "proximity", "suppressed_only", "untuned"]}
    acc["mixed"] = np.concatenate([acc["map"], acc["proximity"]])
    p = [ranksum(acc["proximity"][:, k], acc["map"][:, k]) for k in range(9)]
    q = prv.bh(p)
    print(f"\n{NAMES[r]}: n map {len(acc['map'])}, proximity {len(acc['proximity'])}")
    tab = pd.DataFrame({"proximity": acc["proximity"].mean(0), "map": acc["map"].mean(0), "q": q}, index=prv.CENTERS)
    tab["prox_ci"] = [prv.boot_ci_half(acc["proximity"][:, k], rng_sn, 2000) for k in range(9)]
    tab["map_ci"] = [prv.boot_ci_half(acc["map"][:, k], rng_sn, 2000) for k in range(9)]
    print(tab.round(4).T.to_string())
    dir_ok = tab["proximity"][7] > tab["map"][7] and all(tab["map"][d] > tab["proximity"][d] for d in [11, 13, 15, 17])
    q_ok = all(tab["q"][d] < 0.05 for d in [7, 11, 13, 15, 17])
    pop = {n: summary(r, "10fold", n)[0].mean() for n in ["map", "proximity", "mixed", "suppressed_only", "untuned"]}
    below = {n: (round(acc[n].mean(), 1), round(pop[n], 1)) for n in pop}
    vals = dict(prox7=tab["proximity"][7], map7=tab["map"][7], map15=tab["map"][15], prox15=tab["proximity"][15])
    within = {k: abs(vals[k] - EXP_SN[k][0]) <= EXP_SN[k][1] for k in vals}
    print("directions hold:", dir_ok, "| q < 0.05 at 7, 11, 13, 15, 17:", q_ok,
          "| single-neuron vs population mean accuracy (10-fold):", below,
          "| all below:", all(a < b for a, b in below.values()))
    print("values:", {k: round(v, 2) for k, v in vals.items()}, "within tolerance:", within)
print("\npaper: proximity 7 mm 65.84 +/- 2.582, map 7 mm 59.08 +/- 3.702, map 15 mm 34.72 +/- 4.463, proximity 15 mm 18.84 +/- 2.154")

**Result: reproduced.** Under the BH-FDR classification (120 map, 271 proximity units) proximity units beat map units at 7 mm (65.68 against 59.20%, q = 0.0158) and map units beat proximity units at 11, 13, 15 and 17 mm (q at most 0.0412), every single-neuron group is below its population decoder, and the four values (65.68, 59.20, 35.28 and 19.57%) are within tolerance. Under the uncorrected variant the directions hold but q is 0.2623 at 7 mm and 0.7226 at 11 mm, and the map accuracy at 15 mm (29.87%) is outside 34.72 +/- 4.463.

## figs5de-svm-robust

> The ranking of subpopulations is robust to the decoder; with a linear SVM the map group still has
> the highest average accuracy (72.1%) and proximity and combined groups are best at the closest
> distance.

Linear SVM (scikit-learn SVC, which wraps LIBSVM, linear kernel, C = 1, one-versus-one), with the
preprocessing, trial matching and cross-validation of the MCC decoder. To fit the time limit the
SVM is trained and tested on every third 150-ms window (non-overlapping windows starting 3.0, 3.15,
..., 10.95 s); the MCC results above are recomputed on the same thinned bins to show that thinning
does not move the averages. Both classifications are run with both cross-validations.

In [ ]:
rng_svm = np.random.default_rng(601)
thin = 3
print("MCC overall accuracy, all bins vs every third bin (BH-FDR classification, 10-fold):",
      {n: (round(summary("c", "10fold", n)[0].mean(), 2), round(summary("c", "10fold", n, thin=thin)[0].mean(), 2))
       for n in ["map", "mixed", "proximity", "suppressed_only"]})
SVM_GROUPS = ["map", "mixed", "proximity", "suppressed_only"]
jobs, keys = [], []
for r, cvs in [("m", ["10fold", "15split"]), ("c", ["15split", "10fold"])]:
    g = group_units(r)
    for cv in cvs:
        for name in SVM_GROUPS:
            for s in range(10):
                sel = draw(g, name, rng_svm)
                jobs.append(([[x[:, ::thin] for x in D[i]] for i in sel], int(rng_svm.integers(2**31)),
                             dict(cv=cv, classifier="svm")))
                keys.append((r, cv, name))
t0 = time.time()
with cf.ProcessPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
    out = list(ex.map(prv.decode_job, jobs, chunksize=1))
print(f"{len(jobs)} SVM subsamples in {time.time() - t0:.0f} s on {os.cpu_count()} processes")
SVM = collections.defaultdict(list)
for k, cm in zip(keys, out):
    SVM[k].append(prv.summarize_cm(cm, lefts[::thin])[0])
EXP_SVM = dict(map=(72.14, 7.21), mixed=(54.43, 5.44), proximity=(52.79, 5.28), suppressed_only=(35.01, 3.5))
rows = []
for (r, cv) in sorted({(k[0], k[1]) for k in SVM}):
    acc = {n: np.array(SVM[(r, cv, n)]) for n in SVM_GROUPS}
    m = {n: acc[n].mean(0) for n in SVM_GROUPS}
    ov = {n: m[n].mean() for n in SVM_GROUPS}
    within = {n: abs(ov[n] - EXP_SVM[n][0]) <= EXP_SVM[n][1] for n in SVM_GROUPS}
    rows.append(dict(classification=NAMES[r], cv=cv, **{n: round(ov[n], 2) for n in SVM_GROUPS},
                     map_highest=max(ov, key=ov.get) == "map",
                     prox_and_mixed_beat_map_at_7=bool(m["proximity"][0] > m["map"][0] and m["mixed"][0] > m["map"][0]),
                     map_highest_13_23=f"{sum(all(m['map'][k] > m[n][k] for n in SVM_GROUPS if n != 'map') for k in range(3, 9))}/6",
                     n_within=sum(within.values())))
    print(NAMES[r], cv, "per-distance SVM accuracy:\n", pd.DataFrame({n: np.round(m[n], 1) for n in SVM_GROUPS}, index=prv.CENTERS).T.to_string())
print("\npaper:", {n: v[0] for n, v in EXP_SVM.items()})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** With the linear SVM, under every reading run, the map group has the highest overall accuracy, the proximity and combined groups beat the map group at 7 mm, and the map group is highest at each of 13 to 23 mm. Under the BH-FDR classification three of the four values are within tolerance with either cross-validation (10-fold: map 70.83, proximity 52.93, suppressed-only 33.85%) and the combined group is above it (61.87% with 10-fold, 61.25% with 15 splits, against 54.43 +/- 5.44). Under the uncorrected variant two of four are within tolerance (combined 57.22 and proximity 49.00%, 10-fold). Thinning the time bins moves the MCC averages by at most 0.26 points.

## Whisker-cutting sessions (claims fig4-cut-baseline-unchanged, fig4b-cut-wall-firing-reduced, fig4b-cut-per-unit-changes)

The processed files hold only the full-array passes (trial_condition is empty). The raw file of
each cutting session lists every wall pass (`wall_pass_epochs`), before and after the cut, on the
same clock. The cut is taken as the longest gap between raw passes; the full-array passes are the
deposit's trials before it, the single-whisker passes the raw passes after it. The post-cut passes
have no wall distance, so only measures pooled over distances can be computed. Units: all units of
these sessions that pass the ISI and firing criteria. The paper's population is the 195 units from
18 mice "demonstrating stable waveform properties both before and after cutting" (Methods,
Isolation of Single Unit Activity); the deposit does not say which units or which mice these are,
and the 25 cutting sessions here hold 384 such units from 25 mice. The three claims below are
therefore recorded as blocked (`metadata_insufficient`); the cells show what the deposit gives on
the larger set, which is not the paper's population.

In [ ]:
t0 = time.time()
cut_sessions = [s for s in sessions if "cutting" in s["meta"]["session_description"]]
def cut_info(s):
    raw = prv.raw_asset_for(assets, s["meta"]["path"])
    allst = prv.raw_pass_starts(raw)
    pre, post, tcut, gap, n_after = prv.split_cut(s["start"], allst)
    return dict(session=s["meta"]["session_id"], subject=s["meta"]["subject"], raw_passes=allst.size, deposit_trials=s["start"].size,
                pre=pre.size, post=post.size, cut_at_min=round(tcut / 60, 1), gap_min=round(gap / 60, 1),
                deposit_trials_after_gap=n_after, note=s["meta"]["session_description"].split("Notes:")[-1].strip()[:45],
                pre_starts=pre, post_starts=post)
with cf.ThreadPoolExecutor(8) as ex:
    CUT = list(ex.map(cut_info, cut_sessions))
print(f"read pass epochs of {len(CUT)} raw files in {time.time() - t0:.0f} s")
print(pd.DataFrame(CUT).drop(columns=["pre_starts", "post_starts"]).to_string(index=False))

In 23 of 25 sessions the deposit's trials end before the longest gap, consistent with the cutting
times where the notes give them. In 20230711_sp_session_1 and 20230608_sp_session_1 the deposit's 200 full-array
trials run past the gap (20 and 50 of them come after it, matching raw passes 21 to 220 and 51 to
250), while the notes put the cutting in that gap (49:18-57:15 and 51-62 min). Either the trials
table of these two sessions includes single-whisker passes, or the notes are wrong. Those trials are
dropped from the full-array set here (and are kept, as deposited, in the classification above).

In [ ]:
rows = []
for c in CUT:
    s = [x for x in sessions if x["meta"]["session_id"] == c["session"]][0]
    for i, sp in enumerate(s["spikes"]):
        gi = U.index[(U.session == c["session"])][i]
        if not U.loc[gi, "acc"]:
            continue
        pre_w = prv.counts_in(sp, c["pre_starts"], 3, 11) / 8
        post_w = prv.counts_in(sp, c["post_starts"], 3, 11) / 8
        pre_b = prv.counts_in(sp, c["pre_starts"], -4.5, 0) / 4.5
        post_b = prv.counts_in(sp, c["post_starts"], -4.5, 0) / 4.5
        rows.append(dict(gi=gi, session=c["session"], subject=c["subject"], pre_base=pre_b.mean(), post_base=post_b.mean(),
                         pre_base_perdist=np.mean([np.mean(x) for x in units[gi]["rates"]["base"]]),
                         pre_wall=pre_w.mean(), post_wall=post_w.mean(), pre_w=pre_w, post_w=post_w,
                         m_act=U.loc[gi, "m_act"], c_act=U.loc[gi, "c_act"], tuned=U.loc[gi, "tuned"],
                         m_cat=U.loc[gi, "m_cat"], c_cat=U.loc[gi, "c_cat"]))
CU = pd.DataFrame(rows)
print("units in cutting sessions passing ISI and firing criteria:", len(CU), "from", CU.subject.nunique(), "mice")

## fig4-cut-baseline-unchanged

> Cutting all whiskers but one does not change baseline firing across the recorded population
> (7.09 against 7.10 sp/s; 195 units from 18 mice).

A claim of absence: two-sided Wilcoxon signed-rank test of per-unit mean baseline rates (-4.5 to
0 s) between conditions, at p < 0.05.

In [ ]:
rng_cut = np.random.default_rng(701)
def cut_baseline(df, label):
    w = stats.wilcoxon(df.pre_base, df.post_base)
    print(f"{label}: n={len(df)} units, {df.subject.nunique()} mice; full array {df.pre_base.mean():.2f} +/- "
          f"{prv.boot_ci_half(df.pre_base.values, rng_cut):.3f}, single whisker {df.post_base.mean():.2f} +/- "
          f"{prv.boot_ci_half(df.post_base.values, rng_cut):.3f} sp/s; difference {df.post_base.mean() - df.pre_base.mean():+.2f}; "
          f"signed-rank p={w.pvalue:.3g}")
cut_baseline(CU, "all units")
w2 = stats.wilcoxon(CU.pre_base_perdist, CU.post_base)
print(f"variant, full-array baseline as the mean of per-distance means (code): {CU.pre_base_perdist.mean():.2f} sp/s, p={w2.pvalue:.3g}")
for r in "mc":
    cut_baseline(CU[CU[f"{r}_cat"] != "unresponsive"], f"wall-responsive units only ({NAMES[r]})")
print("\nper mouse (all units): median post - pre baseline difference")
print(CU.assign(d=CU.post_base - CU.pre_base).groupby("subject").d.agg(["count", "median"]).round(2).T.to_string())
print("paper: 195 units from 18 mice; 7.09 +/- 1.034 vs 7.10 +/- 1.038 sp/s; p = 0.70")

**Result: blocked (metadata_insufficient).** The paper's 195 stable units from 18 mice cannot be identified in the deposit. On all 384 units of the 25 cutting sessions, baseline falls from 6.41 to 4.74 sp/s after the cut (signed-rank p = 1.41e-14); on the 283 wall-responsive units of the BH-FDR reading it falls from 6.20 to 4.67 sp/s (p = 5.99e-10; uncorrected variant 362 units, p = 9.78e-14). This set is about twice the paper's and may include the units the paper dropped for unstable waveforms, so it does not decide the claim.

## fig4b-cut-wall-firing-reduced

> In the 86 wall-activated, distance-tuned units, firing during wall passes falls after cutting to a
> single whisker (11.58 against 8.18 sp/s), in many cases to below 1 sp/s.

Units activated and distance-tuned in the full-array condition. Reading (b), mean over all passes
(Methods), is the only one computable; reading (a), the median over distances (code), needs the
post-cut distances.

In [ ]:
rng_cut2 = np.random.default_rng(702)
for r in "mc":
    df = CU[CU[f"{r}_act"] & CU.tuned & (CU[f"{r}_cat"] != "unresponsive")]
    w = stats.wilcoxon(df.pre_wall, df.post_wall)
    print(f"{NAMES[r]}: n={len(df)} units from {df.subject.nunique()} mice; full array {df.pre_wall.mean():.2f} +/- "
          f"{prv.boot_ci_half(df.pre_wall.values, rng_cut2):.3f}, single whisker {df.post_wall.mean():.2f} +/- "
          f"{prv.boot_ci_half(df.post_wall.values, rng_cut2):.3f} sp/s; signed-rank p={w.pvalue:.2e}; "
          f"units below 1 sp/s after cutting: {int((df.post_wall < 1).sum())}; lower after: {int((df.post_wall < df.pre_wall).sum())}")
print("paper: n = 86; 11.58 +/- 1.338 (legend 2.986) vs 8.18 +/- 2.602 sp/s; p = 4.43e-8")

**Result: blocked (metadata_insufficient).** The paper's 86 activated, tuned units among its 195 stable units cannot be identified. On the 175 activated, tuned units of all cutting sessions under the BH-FDR reading, wall firing falls from 10.32 to 6.43 sp/s (signed-rank p = 1.77e-16), and 50 units fall below 1 sp/s; the uncorrected variant gives 198 units, 9.78 against 6.17 sp/s.

## fig4b-cut-per-unit-changes

> Of the 86 activated units, 66 change firing significantly after cutting, 52 decreasing and 14
> increasing.

Per unit, two-sided signed-rank test of per-pass wall rates, passes paired in order and truncated to
the shorter condition, direction from the sign of the median difference. Readings: (a) uncorrected
p < 0.05, (b) BH-FDR across units, (c) Bonferroni, p < 0.05 / (2n). Pass set: (i) all passes pooled
over distances; (ii), the code's per-repetition block, needs the post-cut distances and is not run.

In [ ]:
for r in "mc":
    df = CU[CU[f"{r}_act"] & CU.tuned & (CU[f"{r}_cat"] != "unresponsive")]
    ps, dirs = [], []
    for _, row in df.iterrows():
        n = min(row.pre_w.size, row.post_w.size)
        ps.append(prv.signrank(row.post_w[:n], row.pre_w[:n]))
        dirs.append(np.sign(np.median(row.post_w[:n] - row.pre_w[:n])))
    ps, dirs = np.array(ps), np.array(dirs)
    n = len(df)
    for name, sig in [("a uncorrected", ps < 0.05), ("b BH-FDR", prv.bh(ps) < 0.05), ("c Bonferroni 0.05/(2n)", ps < 0.05 / (2 * n))]:
        print(f"{NAMES[r]} (n={n}), {name}: changed {int(sig.sum())}, decreased {int((sig & (dirs < 0)).sum())}, "
              f"increased {int((sig & (dirs > 0)).sum())}, significant with zero median difference {int((sig & (dirs == 0)).sum())}")
print("paper: 66 changed, 52 decreased, 14 increased of 86; tolerances 6.6, 5.2, 1.4")

**Result: blocked (metadata_insufficient).** On the 175 activated, tuned units of all cutting sessions under the BH-FDR reading (not the paper's 86), 140 change with uncorrected p < 0.05 (114 down, 17 up), 139 with BH-FDR and 118 with Bonferroni (105 down, 9 up). The uncorrected classification gives 198 units, of which 160, 159 and 134 change.

## fig6-model-telc-loses-map (`elsewhere`: the paper's model code)

> A feedforward model in which PrV neurons sum sparse sigmoidal TG excitation and SpVi-relayed
> inhibition yields both proximity-like and map-like tuning across 1,225 model neurons; halving
> long-range inhibition loses map units and expands proximity units.

`PrV_simulation/simulate_prv.py` at commit b1a9f3c, fetched from GitHub, run with its defaults and
`--telc` (control and TeLC are simulated in the same loop, from seed 42); its own `analysis`
classifies the curves. Plotting is replaced by a capture of what the code would plot.

In [ ]:
mods = prv.load_model_code()
outdir = tempfile.mkdtemp()
MODEL = prv.run_model(mods, outdir, telc=True)
for k, v in MODEL.items():
    prox, mp, sup, unt = v["counts"]
    tuned = prox + mp
    print(f"{k}: model neurons {sum(v['counts'])}; proximity {prox}, map {mp}, suppressed {sup}, untuned {unt}; "
          f"distance-tuned (proximity + map, the curves the code keeps) {v['n_selected']}; map share of tuned {100 * mp / tuned:.1f}%, "
          f"proximity share {100 * prox / tuned:.1f}%")
print("paper: 1225 neurons; tuned 590 (control) and 636 (50% inhibition), tolerances 59 and 63.6")

**Result: reproduced.** The paper's model at commit b1a9f3c gives 1,225 neurons, 593 distance-tuned (399 proximity, 194 map) in control and 631 (526 proximity, 105 map) with long-range inhibition halved, both within tolerance of 590 and 636. The map share of tuned neurons falls from 32.7% to 16.6% and the proximity share rises from 67.3% to 83.4%.

## Blocked claims: what the dataset lacks

The cell below lists what one raw file holds and shows the raw wall-actuator line around passes at
different distances. The analog `zaber` line is described in the file as "the only raw record of
wall distance".

In [ ]:
import h5py, remfile
raw = prv.raw_asset_for(assets, "sub-Ephys33/sub-Ephys33_ses-20230602-sp-session-1.nwb")
f = h5py.File(remfile.File(prv.asset_url(raw["asset_id"])), "r")
print("raw file", raw["path"], f"{raw['size'] / 1e9:.1f} GB")
print("acquisition:", list(f["acquisition"].keys()), "; processing:", list(f["processing"].keys()),
      "; intervals:", list(f["intervals"].keys()), "; units table:", "units" in f)
z = f["acquisition/AuxiliaryAnalogSeries/data"]
s33 = [s for s in sessions if s["meta"]["session_id"] == "20230602_sp_session_1"][0]
for k in range(4):
    i0 = int(s33["start"][k] * 30000)
    seg = z[i0 - 30000:i0 + 12 * 30000, 0]
    print(f"pass {k}: wall distance {s33['dist'][k]:.1f} mm; zaber line before onset {np.median(seg[:25000]):.0f}, "
          f"during pass {np.median(seg[60000:300000]):.0f} (raw int16 units)")
f.close()
print("\nall strains:", meta.strain.unique().tolist(), "| descriptions naming tungsten, TeLC, GtACR, laser or anesthesia:",
      int(meta.session_description.str.contains("tungsten|TeLC|GtACR|laser|anesth", case=False).sum()))
print("raw files:", sum(a["path"].endswith(".nwb") and "_ecephys" in a["path"] for a in assets),
      f"({sum(a['size'] for a in assets if a['path'].endswith('.nwb') and '_ecephys' in a['path']) / 1e9:.0f} GB);",
      "video files:", sum(a["path"].endswith(".mp4") for a in assets),
      f"({sum(a['size'] for a in assets if a['path'].endswith('.mp4')) / 1e9:.0f} GB)")

The wall-actuator line takes the same two levels at every distance, so it records when the wall
moves but not where; the post-cut passes therefore have no distance anywhere in the dataset. No
file is from a tungsten, TeLC, GtACR or anesthetized recording, and there is no tracked whisker
kinematics, only raw video of the silicon-probe sessions.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| claim | outcome |
|---|---|
| figs1f-angle-awake-vs-anesthetized | blocked, data_missing |
| figs1j-curvature-awake-vs-anesthetized | blocked, data_missing |
| figs1d-bending-larger-closer | blocked, data_missing |
| figs2-tungsten-responsiveness | blocked, data_missing |
| figs2-tungsten-tuning | blocked, data_missing |
| si-unit-yield | reproduced (SNR criterion not applied) |
| fig2-wall-responsive-counts | reproduced |
| fig2-distance-tuned-counts | partially reproduced |
| fig2f-tuning-categories | reproduced |
| figs4a-baseline-activated-lower | reproduced |
| figs4a-baseline-suppressed-similar | reproduced |
| figs4b-activated-increase-suppressed-decrease | partially reproduced |
| figs4c-untuned-no-change | reproduced |
| fig2de-preferred-distances-span | reproduced |
| figs4ef-crossvalidated-stability | partially reproduced |
| figs4j-ventral-location | reproduced |
| figs4l-temporal-windows-map | partially reproduced |
| fig3a-decoding-above-controls | reproduced |
| fig3-overall-decoding-accuracy | reproduced |
| fig3c-decoding-by-distance | partially reproduced |
| fig3c-decoding-error | reproduced |
| figs5c-single-neuron-decoders | reproduced |
| figs5de-svm-robust | partially reproduced |
| fig4-cut-baseline-unchanged | blocked, metadata_insufficient |
| fig4b-cut-wall-firing-reduced | blocked, metadata_insufficient |
| fig4b-cut-per-unit-changes | blocked, metadata_insufficient |
| fig4c-map-units-after-cut | blocked, metadata_insufficient |
| fig4jk-proximity-units-after-cut | blocked, metadata_insufficient |
| figs5fg-cut-decoding | blocked, metadata_insufficient |
| figs6-opto-local-inhibition-gain | blocked, data_missing |
| figs6i-telc-kinematics-unchanged | blocked, data_missing |
| telc-unit-yield | blocked, data_missing |
| fig5d-telc-preferred-closer | blocked, data_missing |
| fig5e-telc-category-shift | blocked, data_missing |
| fig5e-telc-untuned-ambiguous-unchanged | blocked, data_missing |
| figs6j-telc-untuned-higher-firing | blocked, data_missing |
| figs6k-telc-tuned-baseline-similar | blocked, data_missing |
| figs6k-telc-tuned-wall-higher | blocked, data_missing |
| figs6l-telc-increase-at-close-distances | blocked, data_missing |
| fig6-model-telc-loses-map | reproduced (paper's model code, `elsewhere`) |
| fig6ghi-tuning-shape-telc | blocked, data_missing |

How the analysis differs from the paper's. The SNR isolation criterion is not applied (no
waveforms in the processed files); units are those passing the ISI and firing criteria (1,160
against the paper's 1,141). Responsiveness is judged with BH-FDR across the nine distances, the
claims file's main reading after the answer in questions/001687.yaml; it gives the paper's counts
and the deposit's labels. The uncorrected test, the variant, marks most units responsive; every
result is shown under both. Distance bins pool all passes inside each 2-mm bin (the deposit's
curves use one distance per bin in the two 1-mm-step sessions). Paired tests truncate each bin to
the smallest number of passes, as the code does for two sessions. The decoders are written here
from the Methods and the code (the Neural Decoding Toolbox is MATLAB); the SVM uses scikit-learn's
LIBSVM wrapper on every third 150-ms window. Unit depth uses mean waveforms over 16 raw chunks per
session, not all spikes. The whisker-cutting claims could not be tested on the paper's units.

What was found about the dataset. The deposit holds the wild-type silicon-probe recordings
only: no tungsten, TeLC, GtACR or anesthetized sessions and no tracked whisker kinematics, although
its description names local and long-range inhibition perturbations. The processed files carry no
isolation metrics, no waveforms and no condition labels; the post-cut passes of the 25
whisker-cutting sessions are in the raw files' pass epochs without wall distance, and the raw
wall-actuator line does not encode distance. In two cutting sessions the full-array trials table
includes 20 and 50 passes recorded after the gap in which the notes place the cut. The deposit's
labels follow the BH-FDR test of the code; the Methods do not say the test is corrected. Four files say
they are the same recording as another session; none of those partners is deposited, and no
spike train or pass sequence recurs across files.